In [10]:
import json

import torch
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
import warnings

import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from recsysconfident.environment import Environment
from recsysconfident.setup import Setup
from recsysconfident.ml.models.distribution_based.pr_gat import PRGAT

In [ ]:
def perform_cosine_kmeans(e_emb, min_k=3, max_k=10):
    warnings.filterwarnings('ignore', category=FutureWarning)

    normalized_embeddings = normalize(e_emb, norm='l2')

    best_k = min_k
    best_score = -1
    
    # Needs at least 2 clusters and more data points than clusters
    if e_emb.shape[0] < max_k or e_emb.shape[0] < 2:
        max_k = e_emb.shape[0] - 1
        if max_k < min_k:
            best_k = min_k
            print(f"Warning: Not enough data points to test k values. Using k={best_k}.")
            
    if max_k > min_k:
        print("Finding optimal K using Silhouette Score...")
        for k in range(min_k, max_k + 1):
            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
            cluster_labels = kmeans.fit_predict(normalized_embeddings)
            
            # Check for single cluster/label error
            if len(np.unique(cluster_labels)) > 1:
                score = silhouette_score(normalized_embeddings, cluster_labels)
                if score > best_score:
                    best_score = score
                    best_k = k
                print(f"K={k}, Silhouette Score: {score:.4f}")
        
    print(f"\nOptimal K found: {best_k} (Score: {best_score:.4f})")

    final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
    final_cluster_labels = final_kmeans.fit_predict(normalized_embeddings)
    
    return final_cluster_labels, best_k, final_kmeans


def get_cluster_means(user_emb: np.ndarray, cluster_labels: np.ndarray, n_clusters: int) -> np.ndarray:
    """
    Calculates the mean embedding for each cluster by averaging the original 
    user embeddings belonging to that cluster.
    """
    
    # Initialize the output matrix: (n_clusters, emb_size)
    n_users, emb_size = user_emb.shape
    cluster_means_matrix = np.zeros((n_clusters, emb_size))

    for i in range(n_clusters):
        mask = (cluster_labels == i)
        
        cluster_embeddings = user_emb[mask]
        
        if cluster_embeddings.size > 0:
            cluster_means_matrix[i, :] = cluster_embeddings.mean(axis=0)
        # If a cluster is empty (e.g., from a poor K initialization), 
        # the row remains zeroed out, which is a sensible default.
        
    return cluster_means_matrix

def entity_cluster_means(e_emb):
    cluster_labels, k, k_means = perform_cosine_kmeans(e_emb, min_k=3, max_k=8)
    return get_cluster_means(e_emb, cluster_labels, k), k_means

def entity_similarity_cluster(e_emb, idx, cluster_e_means, e_kmeans):

    group = e_kmeans.predict(e_emb[idx])
    user_group_means = cluster_e_means[group]

    batch_e_emb = e_emb[idx].float()

    norm_batch_users = normalize(batch_e_emb, norm='l2')
    norm_user_group_means = normalize(user_group_means, norm='l2')
    
    user_mean_similarity = torch.sum(norm_batch_users * norm_user_group_means, dim=1)

    return user_mean_similarity

def plot_kde(x, y):

    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)

    # Sort points by density so that high density plots on top
    idx = z.argsort()
    x, y, z = x[idx], y[idx], z[idx]

    # Create scatter plot with density colormap
    plt.scatter(x, y, c=z, s=10, cmap='viridis')
    plt.colorbar(label='Density')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title('Scatter Plot with Point Density')
    plt.show()


In [ ]:
run_folder = "amazon-movies-tvs-prgat-0"

with open(f"../runs/proposal/{run_folder}/setup-0.json", 'r') as f:
    setup_data = json.load(f)
    setup = Setup(**setup_data)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

environ = Environment(model_name=setup.model_name,
                          database_name=setup.database_name,
                          instance_dir=setup.instance_dir,
                          split_position=setup.split_position,
                          batch_size=setup.batch_size,
                          min_inter_per_user=setup.min_inter_per_user,
                          root_path="../"
                          )

model, fit_dl, val_dl, test_dl = environ.get_model_dataloaders(False)
model: PRGAT = model
model.eval()

In [ ]:
users_emb = model.ui_lookup[:model.n_users].detach().cpu().numpy()
items_emb = model.ui_lookup[model.n_users:].detach().cpu().numpy()

cluster_u_means, u_kmeans = entity_cluster_means(users_emb)
cluster_i_means, i_kmeans = entity_cluster_means(items_emb)

In [ ]:
confs = np.array([])
u_sim = np.array([])
i_sim = np.array([])

for data in test_dl:

    users_ids, items_ids, labels = data
    
    u_mean_similarity = entity_similarity_cluster(users_emb, users_ids, cluster_u_means, u_kmeans)
    i_mean_similarity = entity_similarity_cluster(items_emb, items_ids, cluster_i_means, i_kmeans)

    output, mconfs = model.predict(users_ids, items_ids)
    confs = np.concatenate((confs, mconfs), axis=0)
    u_sim = np.concatenate((u_sim, u_mean_similarity))
    i_sim = np.concatenate((i_sim, i_mean_similarity))
    

In [ ]:
conf_u_cluster_sim_corr = np.corrcoef(confs, u_sim)[0, 1]
conf_i_cluster_sim_corr = np.corrcoef(confs, i_sim)[0, 1]

print(f"conf_u_cluster_sim_corr: {conf_u_cluster_sim_corr}")
print(f"conf_i_cluster_sim_corr: {conf_i_cluster_sim_corr}")

In [ ]:
plot_kde(confs, u_sim)

In [ ]:
plot_kde(confs, i_sim)